# 2 · Building and running a graph

A graph is a graph record: its nodes, each naming the node definition it uses and what each of its inputs gets, and the outputs the graph returns. An input gets one of three things: the edges into it, a constant (the same on every call of every run), or a graph input (a value the caller gives, with a default the caller may change). An input with none of them gets the function's own default. `Graph.compile` compiles the record against a registry into a `Graph` and never raises for a fault in it. `graph.run()` calls every node that is ready until the run is complete, waits, fails or is stopped, and returns the `Run`; `run.stream()` yields an event as each node starts and finishes.

In [ ]:
from typing import Annotated

from conductor import Source, Graph, GraphRecord, NodeRecord, NodeRegistry, InputEntry, Param, Series, make_graph
from conductor.widgets import TextInput

registry = NodeRegistry(category="text")


@registry.node("echo", title="Echo", description="Returns its input")
def echo(text: Annotated[str, Param(title="Input", widget=TextInput())]) -> Annotated[str, Param(title="Output")]:
    return text


@registry.node("upper", title="Uppercase", description="Uppercases a text")
def upper(text: Annotated[str, Param(title="Input", widget=TextInput())]) -> Annotated[str, Param(title="Output")]:
    return text.upper()


@registry.node("pair", title="Pair", description="Joins two texts")
def pair(
    a: Annotated[str, Param(title="A", widget=TextInput())],
    b: Annotated[str, Param(title="B", widget=TextInput())],
    separator: Annotated[str, Param(title="Separator", widget=TextInput())] = " + ",
) -> Annotated[str, Param(title="Result")]:
    return f"{a}{separator}{b}"


@registry.node("words", title="Words", description="Gives the words of a text")
def words(text: Annotated[str, Param(title="Input", widget=TextInput())]) -> Annotated[Series[str], Param(title="Words")]:
    return text.split()


@registry.node("join-all", title="Join all", description="Joins every text it is given")
def join_all(texts: Annotated[Series[str], Param(title="Texts")]) -> Annotated[str, Param(title="Result")]:
    return " ".join(texts)

## Build the graph

```text
  hello ──> shout ──┐
                    ├──> pair
  world ────────────┘
```

In [ ]:
record = GraphRecord(
    nodes={
        "hello": NodeRecord(node="echo", inputs={"text": InputEntry(graph_input="hello", default="hello")}),
        "world": NodeRecord(node="echo", inputs={"text": InputEntry(graph_input="world", default="world")}),
        "shout": NodeRecord(node="upper", inputs={"text": InputEntry(sources=(Source("hello", "result"),))}),
        "pair": NodeRecord(node="pair", inputs={
            "a": InputEntry(sources=(Source("shout", "result"),)),
            "b": InputEntry(sources=(Source("world", "result"),)),
            "separator": InputEntry(constant=" & "),
        }),
    },
    outputs={"result": Source("pair", "result")},
)

graph = Graph.compile(record, registry)
print("runnable:", graph.is_runnable)
print("plan:    ", {node_id: described.runs for node_id, described in graph.describe().nodes.items()})

## Ask the compiled graph

`graph.inputs` and `graph.outputs` say what it takes and returns, and `graph.describe()` is all of it as one record: each node's inputs and outputs, and its `runs`, how often it runs (once, or once per item of a series), which `model_dump(mode=\"json\")` writes as JSON.

In [ ]:
pair_inputs = graph.describe().nodes["pair"].inputs
print("pair takes:", [(each.field.name, each.field.type) for each in pair_inputs])

## Problems, not exceptions

A broken graph still compiles. Each problem has a stable `type` and a `loc`, the node and input it is on, and running a graph with one raises `InvalidCallError`.

In [ ]:
broken = Graph.compile(GraphRecord(nodes={"shout": NodeRecord(node="upper", inputs={"text": InputEntry(sources=(Source("ghost", "result"),))})}), registry)
print("runnable:", broken.is_runnable)
for problem in broken.problems:
    print(f"  {problem.type} on {problem.loc[0]}: {problem.msg}")

## Run it

`run.results` is `{node_id: {output: value}}`.

In [ ]:
run = graph.run()
for node_id, outputs in run.results.items():
    print(f"  {node_id}: {outputs['result']!r}")

## Fill its inputs, read its outputs

A graph declares what it takes and returns, as a function declares its parameters and its return value: `hello` and `world` are its graph inputs, each with a default, and `result` is its graph output. A caller gives a graph input by its name, and `run.outputs` is what the graph returns, by the output's name.

In [ ]:
print("takes:  ", list(graph.inputs))
print("returns:", graph.outputs)

graph.run(hello="goodbye").outputs

## Watch the events

In [ ]:
async for event in graph.with_inputs().stream():
    match event.type:
        case "node_start":
            print(f"  start     {event.loc[-1]}")
        case "node_complete":
            print(f"  complete  {event.loc[-1]}: ran {event.outcomes} time(s)")
        case "run_end":
            print(f"  done      {event.run.status}")
        case other:
            print(f"  {other}")

## Once per item

`words` returns a `Series[str]`: many texts. `upper` takes one text, so wired to `words` it runs once per word. `join-all` takes a `Series[str]`, so it runs once and gets every result, in order. `graph.describe()` says so before any run. `make_graph` writes a record from Python: with no `edges` the nodes form a chain, each node under its own id. It declares the graph's inputs and outputs too: each required input that gets nothing, here `text`, and each output of a node no edge leaves, here `result`.

In [ ]:
items = Graph.compile(make_graph(words, upper, join_all), registry)
for node_id, described in items.describe().nodes.items():
    print(f"  {node_id}: {described.runs}")

async for event in items.with_inputs(text="one two three").stream():
    if event.type == "node_progress":
        print(f"  {event.loc[-1]}: {event.count.finished} of {event.count.total}")
    elif event.type == "run_end":
        print("  upper:   ", event.run.results["upper"]["result"])
        print("  join-all:", event.run.results["join-all"]["result"])

## Save it

A `GraphRecord` reads back the JSON it wrote; empty fields are left out.

In [ ]:
text = record.model_dump_json()
print(text)
assert GraphRecord.model_validate_json(text) == record